# figure-per-cell sandbox


In [ ]:
# Если фигуры не отображаются (рендерер IDE не показывает ipympl-виджеты),
# замени widget на inline — статично, но видно везде. Живой зум/пан
# гарантированно работает в браузерном Jupyter Lab: uv run jupyter lab
%matplotlib widget
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT))

from experiments.aggregate import paired_against_baseline
from experiments.figures import (ACCENT, ACCENT_ALT, GRID, INK,
                                 INK_MUTED, NEUTRAL, SURFACE)
from experiments.stats import bootstrap_median_ci

WRITE = False  # True -> save(...) пишет PDF в paper/Image/
IMG = ROOT / "paper" / "Image"


def save(fig, name):
    if not WRITE:
        print(f"предпросмотр: {name} не записан (WRITE=False)")
        return
    fig.savefig(IMG / name, bbox_inches="tight", facecolor=SURFACE)
    print("written:", IMG / name)

## Данные: панельные медианы + CI для сетки и κ-тренда (один раз)


In [5]:
# --- данные для сетки и κ-тренда: медиана + 95% CI на каждую панель ---
# ~15-30 сек, выполняется один раз; дальше только рисовальные ячейки
HOOKS = ["VolatilityHook", "BAHook", "DAHook", "ABHook", "MEVChargeHook"]
REGIMES = ["high", "mid", "low"]
GAS_WEI = [5_000_000_000, 20_000_000_000, 80_000_000_000]
VOLATILE = ["ETH/SHIB", "ETH/USDC"]
RESULT_DIRS = {
    0.03: "results-uu/turnover-0.03",
    0.1: "results-uu/turnover-0.1",
    0.3: "results-uu/turnover-0.3",
    1.0: "results-uu",
    3.0: "results-uu/turnover-3",
}

rows = []
for kappa, rel in RESULT_DIRS.items():
    paired = paired_against_baseline(pd.read_csv(ROOT / rel / "summary.csv"))
    paired = paired[paired["pair"].isin(VOLATILE)]
    for policy in HOOKS:
        for regime in REGIMES:
            for gas in GAS_WEI:
                sub = paired[
                    (paired["policy"] == policy)
                    & (paired["regime"] == regime)
                    & (paired["gas_price_wei"] == gas)
                ]
                d = sub["net_result_delta"]
                lo, hi = bootstrap_median_ci(d)
                rows.append(
                    {
                        "kappa": kappa,
                        "policy": policy,
                        "regime": regime,
                        "gas": gas,
                        "median": float(d.median()),
                        "ci_low": lo,
                        "ci_high": hi,
                        "base_median": float(sub["net_result_baseline"].median()),
                        "n": len(d),
                    }
                )
PANELS = pd.DataFrame(rows)
print(len(PANELS), "панелей готово")

225 панелей готово


## Сетка режим × газ — Fig. 2 (слева) в main; `KAPPA` переключает рабочую точку


In [ ]:
# ================= СЕТКА РЕЖИМ x ГАЗ (Fig. 2 слева в main) =================
plt.close("all")  # прибрать предыдущие открытые фигуры
KAPPA = [0.1, 1.0]  # 0.03 / 0.1 / 0.3 / 1.0 / 3.0
RELATIVE = False  # True: Δ in % of median baseline
# panels; False: absolute USDT/window
PREFIX = {1.0: "uu_", 0.3: "uu_t03_", 0.1: "uu_t01_", 0.03: "uu_t003_", 3.0: "uu_t3_"}

FIGSIZE = (5.3, 4.5)
FS_TICK = 9
FS_POLICY = 9.5
FS_ROWLABEL = 10.5
FS_COLTITLE = 10.5
FS_XLABEL = 9.5
MS, LW_CI = 5.0, 1.6

for kappa in KAPPA:
    t = PANELS[PANELS["kappa"] == kappa]
    fig, axes = plt.subplots(3, 3, figsize=FIGSIZE, sharex=True, sharey=True)
    fig.patch.set_facecolor(SURFACE)
    ys = list(range(len(HOOKS)))[::-1]
    for i, regime in enumerate(REGIMES):
        for j, gas in enumerate(GAS_WEI):
            ax = axes[i][j]
            ax.set_facecolor(SURFACE)
            ax.axvline(0, color="black", linewidth=0.7, zorder=1)
            panel = t[(t["regime"] == regime) & (t["gas"] == gas)]
            for policy, y in zip(HOOKS, ys):
                r = panel[panel["policy"] == policy].iloc[0]
                if RELATIVE:
                    scale = r["base_median"]
                    if scale <= 0:
                        continue
                    med = 100.0 * r["median"] / scale
                    lo, hi = 100.0 * r["ci_low"] / scale, 100.0 * r["ci_high"] / scale
                else:
                    med, lo, hi = r["median"], r["ci_low"], r["ci_high"]
                wins, loses = lo > 0, hi < 0
                colour = ACCENT if wins else (ACCENT_ALT if loses else NEUTRAL)
                ax.plot(
                    [lo, hi],
                    [y, y],
                    color=colour,
                    linewidth=LW_CI,
                    solid_capstyle="round",
                    zorder=2,
                )
                ax.plot(
                    med,
                    y,
                    marker="o",
                    markersize=MS,
                    markerfacecolor=colour if (wins or loses) else SURFACE,
                    markeredgecolor=colour,
                    markeredgewidth=1.1,
                    zorder=3,
                )
            ax.grid(axis="x", color=GRID, linewidth=0.5, linestyle=(0, (1, 2)))
            ax.set_axisbelow(True)
            for spine in ["top", "right", "left"]:
                ax.spines[spine].set_visible(False)
            ax.spines["bottom"].set_color(GRID)
            ax.tick_params(labelsize=FS_TICK)
            ax.xaxis.set_major_locator(plt.MaxNLocator(4))
            if RELATIVE:
                ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}%"))
            else:
                ax.xaxis.set_major_formatter(
                    plt.FuncFormatter(lambda v, _: f"{v / 1000:g}k" if v else "0")
                )
            if j == 0:
                ax.set_yticks(ys)
                ax.set_yticklabels(HOOKS, fontsize=FS_POLICY, color=INK)
                ax.set_ylabel(
                    f"{regime} volatility", fontsize=FS_ROWLABEL, color=INK, labelpad=8
                )
            if i == 0:
                ax.set_title(f"{gas // 10**9} gwei", fontsize=FS_COLTITLE, color=INK)
            if i == 2 and j == 1:
                ax.set_xlabel(
                    "median Δ vs static 30 bps, % of the panel's median baseline result"
                    if RELATIVE
                    else "median Δ vs static 30 bps, USDT/window",
                    fontsize=FS_XLABEL,
                    color=INK,
                )
            ax.margins(y=0.18)
    fig.tight_layout()
    save(fig, f"{PREFIX[kappa]}regime_gas_grid{'_rel' if RELATIVE else ''}.pdf")

## κ-тренд в шторм по газу — Fig. 2 (справа) в main


In [ ]:
# ============== κ-ТРЕНД, ШТОРМОВОЙ СРЕЗ (Fig. 2 справа в main) ==============
plt.close("all")  # прибрать предыдущие открытые фигуры
RELATIVE = False  # True: Δ в % медианного результата baseline точки
# (κ, газ, high-режим); False: абсолютные USDT/window

FIGSIZE = (4.9, 6.8)
FS_TICK, FS_POLICY, FS_XLABEL, FS_SUPY, FS_LEGEND = 9, 9.5, 9, 9, 9
MS, LW_CI = 5.0, 1.8
YLIM = None  # напр. (-60, 120): обрезать раздутые % малых κ,
# где знаменатель (результат baseline) крошечный
SCENARIO_COLOUR = {
    5_000_000_000: "#7fb2e8",
    20_000_000_000: ACCENT,
    80_000_000_000: "#123c6e",
}
OFFSET = {5_000_000_000: 0.88, 20_000_000_000: 1.0, 80_000_000_000: 1.14}
ARB_SHARE = {0.03: "89%", 0.1: "58%", 0.3: "25%", 1.0: "13%", 3.0: "16%"}


def rel_row(r):
    """(median, lo, hi) точки — в %, если RELATIVE, иначе в USDT.
    Неположительный медианный baseline делает долю бессмысленной (знак
    переворачивается) — такая точка отсутствует, а не врёт."""
    if not RELATIVE:
        return r["median"], r["ci_low"], r["ci_high"]
    scale = r["base_median"]
    if scale <= 0:
        return None
    return (
        100.0 * r["median"] / scale,
        100.0 * r["ci_low"] / scale,
        100.0 * r["ci_high"] / scale,
    )


t = PANELS[PANELS["regime"] == "high"]
fig, axes = plt.subplots(len(HOOKS), 1, figsize=FIGSIZE, sharex=True)
fig.patch.set_facecolor(SURFACE)
for ax, policy in zip(axes, HOOKS):
    ax.set_facecolor(SURFACE)
    ax.axhline(0, color="black", linewidth=0.7, zorder=1)
    for gas in GAS_WEI:
        sub = t[(t["policy"] == policy) & (t["gas"] == gas)].sort_values("kappa")
        pts = [
            (k * OFFSET[gas], v)
            for k, v in ((r["kappa"], rel_row(r)) for _, r in sub.iterrows())
            if v is not None
        ]
        if not pts:
            continue
        ax.plot(
            [x for x, _ in pts],
            [v[0] for _, v in pts],
            color=SCENARIO_COLOUR[gas],
            linewidth=1.0,
            alpha=0.5,
            zorder=2,
        )
        for x, (med, lo, hi) in pts:
            filled = lo > 0 or hi < 0
            ax.plot(
                [x, x],
                [lo, hi],
                color=SCENARIO_COLOUR[gas],
                linewidth=LW_CI,
                solid_capstyle="round",
                zorder=3,
            )
            ax.plot(
                x,
                med,
                marker="o",
                markersize=MS,
                markerfacecolor=SCENARIO_COLOUR[gas] if filled else SURFACE,
                markeredgecolor=SCENARIO_COLOUR[gas],
                markeredgewidth=1.1,
                zorder=4,
            )
    ax.set_xscale("log")
    ax.set_ylabel(policy, fontsize=FS_POLICY, color=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.5, linestyle=(0, (1, 2)))
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=FS_TICK)
    if RELATIVE:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}%"))
    else:
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"{v / 1000:g}k" if v else "0")
        )
    ax.margins(y=0.25)
    if RELATIVE and YLIM is not None:
        ax.set_ylim(*YLIM)

kappas = sorted(RESULT_DIRS)
axes[-1].set_xticks(kappas)
axes[-1].set_xticklabels(
    [f"{k:g}\n(arb {ARB_SHARE[k]})" for k in kappas], fontsize=FS_TICK
)
axes[-1].set_xlabel(
    r"$\kappa \text{ — retail turnover, baskets/day}$",
    fontsize=FS_XLABEL,
    color=INK,
)
axes[-1].minorticks_off()
fig.supylabel(
    r"$\text{median } \Delta \text{ vs static 30 bps in high-volatility windows, }$"
    + ("% of the point's median baseline result" if RELATIVE else "USDT/window"),
    fontsize=FS_SUPY,
    color=INK,
)
handles = [
    plt.Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markersize=7,
        markerfacecolor=SCENARIO_COLOUR[g],
        markeredgecolor=SURFACE,
        label=f"{g // 10**9} gwei",
    )
    for g in GAS_WEI
]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=FS_LEGEND)
fig.tight_layout()
save(fig, "uu_kappa_trend_high" + ("_rel" if RELATIVE else "") + ".pdf")

## Данные: трейсы для fee response


In [8]:
# --- данные fee response: трейсы одного окна (быстро) ---
WINDOW_MS = 1_709_251_200_000  # 2024-03-01, штормовой ETH/SHIB день
GAS_FEE = 20_000_000_000
FEE_HOOKS = ["BAHook", "DAHook", "ABHook", "VolatilityHook", "MEVChargeHook"]


def load_swaps(policy):
    path = (
        ROOT
        / "results-uu"
        / f"{policy}-0-ETHUSDT-SHIBUSDT-{WINDOW_MS}-{GAS_FEE}-persistent.jsonl"
    )
    return [r for r in map(json.loads, path.open()) if r.get("kind") == "swap"]


SWAPS = {p: load_swaps(p) for p in FEE_HOOKS}
print({p: len(s) for p, s in SWAPS.items()})

{'BAHook': 837, 'DAHook': 1117, 'ABHook': 1218, 'VolatilityHook': 530, 'MEVChargeHook': 653}


## Поведение комиссий за один день — evaluation-draft


In [ ]:
# ============== ПОВЕДЕНИЕ КОМИССИЙ (fee response, evaluation-draft) ==============
plt.close("all")  # прибрать предыдущие открытые фигуры
FS_TICK, FS_YLABEL, FS_ANNOT, FS_LEGEND, FS_XLABEL, FS_SUPY = 8, 8.5, 8.5, 8, 9, 9
LW_FEE, LW_PRICE = 1.2, 1.3
PANEL_H, PRICE_H = 1.15, 1.5  # высоты панелей (дюймы, множитель)


def style(ax):
    ax.set_facecolor(SURFACE)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(True, axis="y", color=GRID, linewidth=0.5, linestyle=(0, (1, 2)), zorder=0)
    ax.set_axisbelow(True)


price_rows = {}
for s in next(iter(SWAPS.values())):
    price_rows.setdefault(s["candle"], s["extPrice0"] / s["extPrice1"])
prices = pd.Series(price_rows).sort_index()
prices = prices / prices.iloc[0]

fig, axes = plt.subplots(
    len(FEE_HOOKS) + 1,
    1,
    figsize=(7.2, PANEL_H * len(FEE_HOOKS) + 1.9),
    sharex=True,
    gridspec_kw={"height_ratios": [PRICE_H] + [1] * len(FEE_HOOKS)},
)
fig.patch.set_facecolor(SURFACE)

hours = prices.index / 60.0
axes[0].plot(hours, prices.to_numpy(), color=INK, linewidth=LW_PRICE, zorder=3)
axes[0].set_ylabel("price,\nindexed", fontsize=FS_YLABEL, color=INK)
style(axes[0])

for ax, policy in zip(axes[1:], FEE_HOOKS):
    swaps = SWAPS[policy]
    hrs = [s["candle"] / 60.0 for s in swaps]
    ab = [s["feeAB"] / 100.0 for s in swaps]
    ba = [s["feeBA"] / 100.0 for s in swaps]
    ax.axhline(30, color=INK, linewidth=0.8, linestyle=(0, (4, 3)), zorder=2)
    ax.plot(
        hrs,
        ab,
        color=ACCENT,
        linewidth=LW_FEE,
        drawstyle="steps-post",
        zorder=3,
        label=r"fee A$\rightarrow$B",
    )
    ax.plot(
        hrs,
        ba,
        color=ACCENT_ALT,
        linewidth=LW_FEE,
        drawstyle="steps-post",
        zorder=3,
        label=r"fee B$\rightarrow$A",
    )
    ax.set_yscale("log")
    ax.set_ylim(0.7, 1500)
    ax.set_yticks([1, 10, 100, 1000])
    ax.set_yticklabels(["1", "10", "100", "1000"])
    ax.minorticks_off()
    ax.annotate(
        policy,
        (0.012, 0.93),
        xycoords="axes fraction",
        fontsize=FS_ANNOT,
        color=INK,
        va="top",
        zorder=5,
        bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 1.5},
    )
    style(ax)

axes[1].legend(loc="lower left", frameon=True, fontsize=FS_LEGEND, ncol=2)
axes[-1].set_xlabel("hours into the window", fontsize=FS_XLABEL, color=INK)
fig.supylabel(
    "applied fee by direction, basis points (log)", fontsize=FS_SUPY, color=INK
)
fig.align_ylabels()
fig.tight_layout()
save(fig, "uu_fee_response.pdf")

## Данные: интрадей-форма спроса


In [10]:
# --- данные demand calibration: медианный день 2024 (десятки секунд) ---
from experiments.binance import fetch_klines
from experiments.uu import uu_profile

YEAR_START, YEAR_END = 1_704_067_200_000, 1_735_689_600_000
SMOOTH_MIN = 20

df0 = fetch_klines("ETHUSDT", YEAR_START, YEAR_END, ROOT / "data", with_volume=True)
df1 = fetch_klines("SHIBUSDT", YEAR_START, YEAR_END, ROOT / "data", with_volume=True)
n = min(len(df0), len(df1))
prof = uu_profile(df0.iloc[:n].reset_index(), df1.iloc[:n].reset_index())
prof = prof.assign(minute=(prof["open_time"] // 60_000) % 1440)
day = prof.groupby("minute")["v_usdt"].median().to_numpy()
padded = np.concatenate([day] * 3)
day = (
    pd.Series(padded)
    .rolling(SMOOTH_MIN, center=True, min_periods=1)
    .mean()[1440 : 2 * 1440]
    .to_numpy()
)
SHAPE = day / day.sum()  # s-hat медианного дня: сумма = 1
print("shape готов, сумма =", SHAPE.sum())

shape готов, сумма = 1.0


## Калибровка спроса — evaluation-draft


In [ ]:
# ============== КАЛИБРОВКА СПРОСА (demand calibration, evaluation-draft) ==============
plt.close("all")  # прибрать предыдущие открытые фигуры
BASKET = 20_000_000
DEMAND_LEVELS = [  # κ, подпись, измеренная доля арбитража
    (3.0, "flagship pool", "16%"),
    (1.0, "primary venue (calibrated)", "13%"),
    (0.3, "mid-tier pool", "25%"),
    (0.1, "secondary venue", "58%"),
    (0.03, "over-provisioned pool", "89%"),
]
SHADES = ["#123c6e", "#2a78d6", "#5f9be0", "#8ab5e8", "#bcd4f0"]  # сверху вниз

FIGSIZE = (3.5, 2.9)
FS_TICK, FS_LABEL, FS_ANNOT, FS_SESSION = 7.5, 8, 6.4, 6.8
RIGHT = 0.64  # доля ширины под кривые (остальное — подписи справа)

hours = np.arange(1440) / 60.0
fig, ax = plt.subplots(figsize=FIGSIZE)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)

for (kappa, kind, arb), colour in zip(DEMAND_LEVELS, SHADES):
    v = kappa * BASKET * SHAPE
    emphasis = kappa == 1.0
    ax.plot(
        hours,
        v,
        color=colour,
        linewidth=2.0 if emphasis else 1.2,
        zorder=3 if emphasis else 2,
    )
    daily = f"{kappa * BASKET / 1e6:g}M"
    ax.annotate(
        f"κ = {kappa:g} · {daily}/day · arb {arb}\n{kind}",
        (24.15, v[-1]),
        fontsize=FS_ANNOT,
        color=INK if emphasis else INK_MUTED,
        va="center",
        ha="left",
        annotation_clip=False,
        fontweight="bold" if emphasis else "normal",
    )

top = 3.0 * BASKET * SHAPE.max()
ax.set_ylim(0.03 * BASKET * SHAPE.min() * 0.55, top * 3.2)
for x, txt in [(3.2, "Asia (quiet)"), (15.8, "US session")]:
    ax.annotate(txt, (x, top * 1.9), fontsize=FS_SESSION, color=INK, ha="center")

ax.set_yscale("log")
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 6))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 25, 6)], fontsize=FS_TICK)
ax.set_xlabel("time of day (UTC)", fontsize=FS_LABEL, color=INK)
ax.set_ylabel("potential retail demand, USDT/min", fontsize=FS_LABEL, color=INK)
ax.tick_params(labelsize=FS_TICK)
ax.grid(axis="y", color=GRID, linewidth=0.5, linestyle=(0, (1, 2)))
ax.set_axisbelow(True)
fig.tight_layout()
fig.subplots_adjust(right=RIGHT)
save(fig, "uu_demand_calibration.pdf")

## Финализация

```python
WRITE = True
```

(в setup-ячейке или прямо перед save), перезапустить рисовальные ячейки —
для сетки все нужные `KAPPA`. Затем:

```bash
cd paper && latexmk -pdf main.tex && latexmk -pdf evaluation-draft.tex
```
